# Notebook 02 — TA CRM QC and pH-standard QC

**Role in the pipeline:** this is the *first* notebook on the critical path.
Its `derived.csv` (per sheet) is the file that Stage 1A reads.

```text
oa_prelim_data.xlsx
   │
   └──► 02_ta_ph_qc.ipynb  (THIS NOTEBOOK)
           ├─ TA CRM correction
           ├─ pH-standard correction
           └─ derived.csv  ──►  04_stage1a / 05_stage1b / ... / 08_stage4
```

**What it does, per sheet**

1. Detects TA Certified Reference Material (CRM) rows, computes
   `certified − measured` Total Alkalinity, applies the SOP rules
   (`NO_ADJUST` / `ADJUST` / `FAIL` / `INSUFFICIENT_DATA`), and writes
   corrected TA back to the dataframe.
2. Detects pH-standard rows (TRIS / AMP / BIS), interpolates expected pH at
   the measured temperature, computes the residual, classifies as
   `OK` / `WARN` / `FAIL`, and (optionally) corrects sample pH.
3. Writes a `derived.csv` (the QC-corrected sheet), per-QC tables, JPEG
   plots, and one-page markdown reports.
4. Records every input, parameter, output and version in `logs/manifest.json`.

All the heavy QC math lives in `oa_qc_ta_ph.py`; this notebook is a thin
orchestrator. See `02_ta_ph_qc.README.md` for the design rationale and
the source citations behind each choice.


## Parameters

Single tagged `parameters` cell — override from the command line via
Papermill (`papermill 02_ta_ph_qc.ipynb run.ipynb -p XLSX_PATH "..."`) or
edit and Restart-and-Run-All in Jupyter.

The block is long because the QC has many knobs. Sections are grouped by
purpose. If you only want the defaults, you usually only need to change
`XLSX_PATH` and possibly `SHEET`.


In [ ]:
# =====================================================================
# Parameters cell  (papermill tag: "parameters")
# =====================================================================
# All explanations live in the markdown cell directly below this one.
# Keeping this cell free of inline trailing comments avoids the
# Papermill "unknown parameter" warnings that those comments trigger.

# --- I/O -------------------------------------------------------------
from pathlib import Path

# Support kernels started at the repository root or in notebooks/.
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

XLSX_PATH = str(REPO_ROOT / "examples" / "example_data.xlsx")
OUT_DIR = None
SHEET = 0
CONFIG_PATH = "None"

# --- Display / output toggles ---------------------------------------
PREVIEW_ROWS = 15
HTML_MAX_ROWS = None
OPEN_HTML = False
WRITE_DERIVED_CSV = True
WRITE_HTML_TABLE = True
WRITE_PLOTS = True

# --- Column names expected in each sheet ----------------------------
TA_COL = "ta"
PH_COL = "pH_lab"
PH_TEMP_COL = "temp_lab"
SAMPLE_TAG_COL = "sample_tag"

# --- Row roles (ADR-0006) -------------------------------------------
ROW_ROLE_PREFIXES = {"CRM": "RM", "STANDARD": "TRIS"}

# --- TA CRM QC ------------------------------------------------------
CRM_CORRECT_TA = True
CRM_BATCH = "213"
# AUDIT FIX N-1: path to the authoritative certified CRM values file
# (configs/crm_certified_values.yaml, transcribed from the NOAA OCADS
# Dickson CRM batch table). When None, the corrected in-code fallback
# table in roap.qc_ta_ph is used. Prefer the YAML file.
CRM_VALUES_CONFIG = str(REPO_ROOT / "configs" / "crm_certified_values.yaml")
CRM_TA_OVERRIDE = None
NO_REQUIRE_TA_FOR_CRM = False
GROUP_BY = None
MIN_CRM_N = 2
TA_MAD_K = 3.5
TA_MAX_ABS_DIFF = 20.0
CORRECT_CRM_TOO = False
TA_NO_ADJUST = 2.0
TA_REJECT = 20.0
TA_PLOT_TITLE = "RM Alkalinity Difference"
TA_PLOT_NO_LABELS = False

# --- pH standard QC -------------------------------------------------
PHSTD_QC = True
PH_BUFFER = "tris"
PHSTD_CORRECT_SAMPLES = False
MIN_PHSTD_N = 2
PH_MAD_K = 3.5
PH_MAX_ABS_DIFF = 0.03
PH_OK = 0.02
PH_WARN = 0.05
PH_PLOT_TITLE = "TRIS pH Difference"
PH_PLOT_NO_LABELS = False


### Parameter reference

The cell above is the single Papermill-tagged `parameters` cell. Keeping it
free of inline trailing comments avoids spurious *unknown parameter*
warnings from Papermill. The per-parameter notes that used to live next to
each assignment are reproduced below.

**I/O**

- `XLSX_PATH` — full path to the source `.xlsx` workbook.
- `OUT_DIR` — output directory. `None` means use `"<workbook_stem>__qc_outputs/"` next to the workbook.
- `SHEET` — `"0"`, a sheet name, or `"all"`.
- `CONFIG_PATH` — path to a JSON/YAML config to override defaults. The literal string `"None"` (or `None`) means "use built-in defaults".

**Display / output toggles**

- `HTML_MAX_ROWS` — `None` means write all rows to the HTML table.
- `OPEN_HTML` — open the HTML table in a browser after writing.
- `WRITE_DERIVED_CSV` / `WRITE_HTML_TABLE` / `WRITE_PLOTS` — output toggles.

**Row roles**

- `ROW_ROLE_PREFIXES` — Tag prefixes, matched case-insensitively against `SAMPLE_TAG_COL`, that give a row the CRM or Standard role. Every other row, including rows with a blank tag, is a Sample. A config may set one key and keep the other default. The Tag prefix is the only signal (ADR-0006); `sample_type` / `crm_or_sample` columns are not read.

**TA CRM QC**

- `CRM_BATCH` — must be a key in `CRM_CERTIFIED_TA` inside `oa_qc_ta_ph.py`.
- `CRM_TA_OVERRIDE` — numeric value overrides the certified TA; `None` uses the certified value for `CRM_BATCH`.
- `GROUP_BY` — column name to group corrections by, or `None`.
- `MIN_CRM_N` — minimum number of non-outlier CRMs required.
- `TA_MAD_K` — MAD multiplier for the outlier rule.
- `TA_MAX_ABS_DIFF` — absolute-difference outlier cap; `0` disables this rule.
- `CORRECT_CRM_TOO` — if `False` (default), correction is applied to samples only.
- `TA_NO_ADJUST` — SOP "noise floor": `|corr| <= this` is forced to `0`.
- `TA_REJECT` — SOP reject: `|corr| > this` withholds the row and flags `FAIL`.

**pH standard QC**

- `PH_BUFFER` — `"tris"`, `"amp"`, or `"bis"`.
- `PH_MAX_ABS_DIFF` — absolute-difference cap; `0` disables this rule.
- `PH_OK` — status `OK` when `|mean diff| <= this`.
- `PH_WARN` — status `WARN` when `|mean diff| <= this`; `FAIL` otherwise.


## Setup

Dependencies are managed by `pyproject.toml` / Pixi, not installed inside the
notebook. The Stage lives in `roap.qc_ta_ph`: `run(sheets, stage_config,
params)` applies TA CRM QC and pH-standard QC to each sheet and returns a
`StageResult` without touching disk. This notebook resolves the workbook and
config, calls `run`, prints the per-sheet status and writes the result.

In [ ]:
from __future__ import annotations

import webbrowser
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = None

from roap import qc_ta_ph
from roap.common import (
    as_bool,
    die,
    fmt,
    is_none_like,
    normalize_columns,
    print_quick_summary,
    read_excel_sheets,
)
from roap.stage_config import REMOVED_QC_PARAMETERS, load_stage_config
from roap.stage_result import write_stage_result

## Resolve workbook path and load sheets

In [ ]:
# ---- Resolve input workbook first ------------------------------------
xlsx_path = Path(XLSX_PATH).expanduser().resolve()
if not xlsx_path.exists():
    die(f"File not found: {xlsx_path}")
if xlsx_path.suffix.lower() != ".xlsx":
    die(f"Expected .xlsx, got: {xlsx_path.name}")

# ---- Optional per-stage config ---------------------------------------
# The parameters cell holds Stage QC's defaults; the optional config file's
# `parameters:` block overrides them. roap.stage_config checks the file's
# compatible_pipeline_version and rejects unknown or removed keys (#41).
stage_cfg = load_stage_config("qc", CONFIG_PATH)
config_path_resolved = stage_cfg.source
for _warning in stage_cfg.warnings:
    print(f"WARNING: {_warning}")
for key, value in stage_cfg.config["parameters"].items():
    globals()[key] = value

# Papermill injects removed parameters as new globals; reject them too.
# Row roles come from the Tag prefix only (ADR-0006, #40).
for _key in sorted(set(REMOVED_QC_PARAMETERS) & set(globals())):
    die(f"Parameter {_key} was removed: {REMOVED_QC_PARAMETERS[_key]}")

# ---- Normalise and validate the run parameters before any QC work ----
OPEN_HTML = as_bool(OPEN_HTML)
PREVIEW_ROWS = int(PREVIEW_ROWS)
try:
    qc_params = qc_ta_ph.qc_run_parameters({key: globals()[key] for key in qc_ta_ph.QC_RUN_DEFAULTS})
except ValueError as exc:
    die(str(exc))

# ---- Certified CRM values (AUDIT FIX N-1) ----------------------------
# Loaded from the authoritative, versioned YAML (transcribed from the NOAA
# OCADS Dickson batch table) and version-checked (#41). A relative path that
# does not exist from the current directory is tried next to the workbook.
# None uses the corrected in-code fallback table.
if qc_params["CRM_CORRECT_TA"] and not is_none_like(CRM_VALUES_CONFIG):
    _crm_cfg = Path(CRM_VALUES_CONFIG).expanduser()
    if not _crm_cfg.is_absolute() and not _crm_cfg.exists():
        _alt = xlsx_path.parent / CRM_VALUES_CONFIG
        if _alt.exists():
            _crm_cfg = _alt
    crm_values = stage_cfg.load_crm_values(_crm_cfg)
    print(f"Loaded {len(crm_values)} certified CRM values from: {_crm_cfg}")
else:
    print("Using corrected in-code CRM fallback table (CRM_VALUES_CONFIG not set).")

# ---- Sheet selection and output root ---------------------------------
if str(SHEET).strip().lower() == "all":
    sheet_param: "str | int" = "all"
else:
    try:
        sheet_param = int(SHEET)
    except ValueError:
        sheet_param = SHEET

sheets = read_excel_sheets(xlsx_path, sheet_param)

if OUT_DIR:
    out_root = Path(OUT_DIR).expanduser().resolve()
else:
    out_root = xlsx_path.parent / f"{xlsx_path.stem}__qc_outputs"

print(f"Workbook   : {xlsx_path}")
print(f"Sheets     : {list(sheets.keys())}")
print(f"Output root: {out_root}")
print(f"Config     : {config_path_resolved or '(built-in defaults)'}")

## Quick preview of each sheet before QC

In [ ]:
for sheet_name, df in sheets.items():
    df = normalize_columns(df)
    print_quick_summary(df, sheet_name, preview_rows=PREVIEW_ROWS)
    print("\n" + "=" * 80 + "\n")


## Run TA CRM QC + pH-standard QC per sheet

For each sheet `qc_ta_ph.run`:

1. Assigns each row's role from its Tag prefix (ADR-0006).
2. Applies TA CRM correction (if `CRM_CORRECT_TA`).
3. Applies pH standard QC (if `PHSTD_QC`).
4. Returns `data/derived.csv` (the QC-corrected sheet — this is what Stage 1A
   reads), the QC and correction tables, JPEG plots and markdown reports.

Filenames are intentionally short. The sheet identity is in the parent
folder (`sheet_<safe_sheet>/`), so we do not repeat the workbook stem or
"stage 02" tag in every file. The pH-buffer infix (`tris`/`amp`/`bis`) is
kept because it is a meaningful variant — running with two buffers in the
same `out_root` should not overwrite outputs.

In [ ]:
try:
    result = qc_ta_ph.run(sheets, stage_cfg, qc_params, source=str(xlsx_path))
except ValueError as exc:
    die(str(exc))

for sheet_name, info in result.manifest["sheet_summaries"].items():
    print(f"Sheet: {sheet_name}")
    print(f"  Row roles: {info['row_role_counts']}")
    ta = info.get("ta_summary") or {}
    if ta:
        print(
            "  TA CRM:",
            f"status={ta.get('overall_status')},",
            f"detected={ta.get('crm_n_detected')},",
            f"kept={ta.get('crm_n_kept')},",
            f"mean_diff={fmt(ta.get('overall_corr'), nd=3)}",
        )
    ph = info.get("ph_summary") or {}
    if ph:
        print(
            "  pH STD:",
            f"status={ph.get('overall_status')},",
            f"detected={ph.get('n_detected')},",
            f"kept={ph.get('n_kept')},",
            f"mean_diff={fmt(ph.get('mean_diff_kept'), nd=4)}",
        )
        if qc_params["WRITE_PLOTS"] and not any(
            rel.startswith(info["root"] + "/figures/phstd_") for rel in result.figures
        ):
            print(f"  pH STD: no usable standard diffs (kept={ph.get('n_kept')}); no QC plot.")
    print(f"  Safe sheet name: {info['sheet_safe_name']}")

## Write outputs and the manifest

`write_stage_result` writes every sheet's files, a single `logs/manifest.json`
recording the inputs, parameters, outputs and package versions, and the run
bundle, which hashes the workbook, the configs and every file this run
wrote so `roap verify-bundle <OUT_DIR>` can check them later. Provenance
lives here, not in filenames.

In [ ]:
qc_ta_ph.anchor_manifest_paths(result, out_root)  # absolute paths, as before
result.manifest["input_xlsx"] = str(xlsx_path)
result.manifest["parameters"].update(
    {
        "XLSX_PATH": str(xlsx_path),
        "OUT_DIR": str(out_root),
        "SHEET": SHEET,
        "CONFIG_PATH": config_path_resolved,
        "PREVIEW_ROWS": PREVIEW_ROWS,
        "OPEN_HTML": OPEN_HTML,
        "CRM_VALUES_CONFIG": str(CRM_VALUES_CONFIG),
    }
)

written = write_stage_result(result, out_root, parquet=False)
outputs_df = pd.DataFrame({"output_file": [str(p) for p in written.values()]})
print(f"Total files written: {len(outputs_df)}")
if display is not None:
    display(outputs_df)
else:
    print(outputs_df.to_string(index=False))

if OPEN_HTML:
    for rel, path in written.items():
        if rel.endswith("/tables/table.html"):
            webbrowser.open(path.as_uri())